# Evaluate code

In [1]:
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found qwen/qwen3-coder-480b-a35b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(


## Criando dataset no LangSmith

In [2]:
from datasets import load_dataset
from tqdm import tqdm

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)


Problems: 100%|██████████| 164/164 [00:00<00:00, 13742.48problem/s]


In [3]:
from langsmith import Client

client_langsmith = Client()


tamanho_amostra = 5

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(data_set_code_langsmith)), tamanho_amostra)
for index in index_aleatorios:
    data_aleatorios.append(data_set_code_langsmith[index])


# Create dataset if it doesn't exist
if not client_langsmith.has_dataset(dataset_name=dataset_name):
    dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name, 
        description="The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models."
    )
    
    client_langsmith.create_examples(dataset_id=dataset.id, examples=data_aleatorios)

In [4]:
data_aleatorios

[{'inputs': {'aswer_code': '\ndef split_words(txt):\n    \'\'\'\n    Given a string of words, return a list of words split on whitespace, if no whitespaces exists in the text you\n    should split on commas \',\' if no commas exists you should return the number of lower-case letters with odd order in the\n    alphabet, ord(\'a\') = 0, ord(\'b\') = 1, ... ord(\'z\') = 25\n    Examples\n    split_words("Hello world!") ➞ ["Hello", "world!"]\n    split_words("Hello,world!") ➞ ["Hello", "world!"]\n    split_words("abcdef") == 3 \n    \'\'\'\n'},
  'outputs': {'response_code': 'def check(candidate):\n\n    assert candidate("Hello world!") == ["Hello","world!"]\n    assert candidate("Hello,world!") == ["Hello","world!"]\n    assert candidate("Hello world,!") == ["Hello","world,!"]\n    assert candidate("Hello,Hello,world !") == ["Hello,Hello,world","!"]\n    assert candidate("abcdef") == 3\n    assert candidate("aaabb") == 2\n    assert candidate("aaaBb") == 1\n    assert candidate("") == 0\n

In [5]:
dataset = client_langsmith.list_datasets()

### Avaliando o agente

In [6]:
from langsmith import traceable
from langchain_core.messages import HumanMessage

In [7]:
@traceable
async def agent_avaliado(question):
    answer = await agent.ainvoke(
    {
        "messages": 
            [HumanMessage(role="user",
                          content=question)],
        "todos": [],
    }
)
    return answer['messages'][-1].content


In [8]:
# We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics.
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

LLM-as-a-Judge Evaluator
For open-ended metrics, it's can be powerful to use an LLM to score the outputs.

Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [9]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score for the correctness of the answer, of an answer between 0 and 1")

In [10]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

async def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["aswer_code"], outputs["output"], reference_outputs["response_code"])

    
    """model = init_chat_model(model = "moonshotai/kimi-k2-instruct-0905", model_provider = "nvidia")
    
    model_stutured = model.with_structured_output(CorrectnessScore)
    
    response = await model_stutured.ainvoke([HumanMessage(content=prompt)])"""
    
    model = GoogleModel('models/gemini-2.0-flash-lite')
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = agent.run_sync(prompt)
    except Exception as e:
        logger.error(f"Erro do tipo: {e}")
        model = GoogleModel('models/gemini-2.0-flash')
        agent = Agent(model, output_type=CorrectnessScore)
        response = agent.run_sync(prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response

In [11]:
import nest_asyncio

# Apply nest_asyncio at the start of your notebook
nest_asyncio.apply()

def correctness_sync(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return asyncio.run(correctness(inputs, outputs, reference_outputs))

In [12]:
response = correctness_sync({"aswer_code": "def add(a, b):\n    return a + b"}, {"output": "def add(a, b):\n    return a + b"}, {"response_code": "def add(a, b):\n    return a + b"})

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"


In [13]:
response

1

In [14]:
# 4. Define a function to run your application
async def run(inputs: dict):
    return await agent_avaliado(inputs["aswer_code"])


In [15]:
dataset_name

'Human-Eval-Code-5-aleatorios'

In [ ]:
from langsmith import evaluate, aevaluate

modelos = [
    #{"model": "openai/gpt-oss-20b",  "provider": "groq"},
    #{"model": "meta-llama/llama-4-scout-17b-16e-instruct",  "provider": "groq"},
    #{"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "groq"},
    #{"model": "qwen/qwen3-next-80b-a3b-instruct",  "provider": "nvidia"},
    #{"model": "deepseek-ai/deepseek-v3.1",  "provider": "nvidia"},
    {"model": "microsoft/phi-4-mini-instruct",  "provider": "nvidia"},
    {"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "nvidia"},
    {"model": "meta/llama-4-scout-17b-16e-instruct",  "provider": "nvidia"},
    {"model": "openai/gpt-oss-120b",  "provider": "nvidia"},
    {"model": "moonshotai/kimi-k2-instruct",  "provider": "nvidia"},
    {"model": "meta/llama-3.3-70b-instruct",  "provider": "nvidia"},
    {"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",  "provider": "nvidia"},
    {"model": "nvidia/llama-3.1-nemotron-ultra-253b-v1",  "provider": "nvidia"},
    {"model": "meta/llama-3.1-405b-instruct",  "provider": "nvidia"},
    {"model": "nvidia/llama-3.1-nemotron-nano-4b-v1.1",  "provider": "nvidia"},
    {"model": "ibm/granite-3.3-8b-instruct",  "provider": "nvidia"},
    {"model": "nv-mistralai/mistral-nemo-12b-instruct",  "provider": "nvidia"},
    {"model": "nvidia/llama-3.3-nemotron-super-49b-v1",  "provider": "nvidia"},
    {"model": "mistralai/mistral-small-3.1-24b-instruct-2503",  "provider": "nvidia"},
    {"model": "qwen/qwq-32b",  "provider": "nvidia"},
    {"model": "meta/llama-3.1-8b-instruct",  "provider": "nvidia"},
    {"model": "mistralai/mistral-nemotron",  "provider": "nvidia"},
    {"model": "meta/llama-3.2-3b-instruct",  "provider": "nvidia"},
    {"model": "openai/gpt-oss-20b",  "provider": "nvidia"},
    {"model": "mistralai/mistral-large-2-instruct",  "provider": "nvidia"},
    {"model": "deepseek-ai/deepseek-r1-0528",  "provider": "nvidia"},
    {"model": "meta/llama-3.1-70b-instruct",  "provider": "nvidia"}
    
    
]

models_concluidos = []

for modelo in modelos:
    print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provider']}")
    
    if modelo['model'] in models_concluidos:
        print("Modelo já concluído")
        continue
    else:
        code_agent = CodeAgentReact(model=modelo['model'], model_provider=modelo['provider'])
        agent = code_agent.create_agent()
        
        @traceable
        async def agent_avaliado(question):
            answer = await agent.ainvoke(
            {
                "messages": 
                    [HumanMessage(role="user",
                                content=question)],
                "todos": [],
            }
        )
            return answer['messages'][-1].content
        
        async def run(inputs: dict):
            return await agent_avaliado(inputs["aswer_code"])
        
        
        results = asyncio.run(aevaluate(
            run,
            data=dataset_name,
            evaluators=[correctness, conciseness],
            experiment_prefix=f"{dataset_name}-{modelo['model']}-{modelo['provider']}"))
        
        models_concluidos.append(modelo['model']) 

INFO:code_agent.creat_react_code_agent.code_agent_react:CodeAgentReact inicializado: provider=nvidia, checkpointer=False
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso
INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso


Rodando o agente com o modelo microsoft/phi-4-mini-instruct do provedor nvidia
View the evaluation results for experiment: 'Human-Eval-Code-5-aleatorios-microsoft/phi-4-mini-instruct-nvidia-103a58cc' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/4dfd7fc2-d782-4c4d-ba17-87f5a251f9a7/compare?selectedSessions=fd5fe5b9-6915-401d-b758-2f3802325580




0it [00:00, ?it/s]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
1it [00:22, 22.93s/it]

In [ ]:
dataset_name

'Human-Eval-Code-5-aleatorios'

### Avaliando os resultados

In [ ]:
[experiment.name for experiment in client_langsmith.list_projects()]

['Human-Eval-Code-5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-d4a27598',
 'Human-Eval-Code-5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-696fcfd1',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-large-2-instruct-nvidia-e1645f1c',
 'Human-Eval-Code-5-aleatorios-openai/gpt-oss-20b-nvidia-8a4f22b3',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-b3fcb57f',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-7665a03d',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-nemotron-nvidia-c56c3316',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-bc289731',
 'Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvidia-30b9c36d',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-3814948e',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-nemotron-nvidia-18a47005',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-768ce160',
 'Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvidia-6a576a96',
 'Human-Eval-Code-

In [ ]:
experiments_names = [experiment.name for experiment in client_langsmith.list_projects() if "Human-Eval-Code-" in experiment.name]
experiments_names

['Human-Eval-Code-5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-d4a27598',
 'Human-Eval-Code-5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-696fcfd1',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-large-2-instruct-nvidia-e1645f1c',
 'Human-Eval-Code-5-aleatorios-openai/gpt-oss-20b-nvidia-8a4f22b3',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-b3fcb57f',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-7665a03d',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-nemotron-nvidia-c56c3316',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-bc289731',
 'Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvidia-30b9c36d',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-3814948e',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-nemotron-nvidia-18a47005',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-768ce160',
 'Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvidia-6a576a96',
 'Human-Eval-Code-

In [ ]:
# Set this to load expt results
for experiment_name in experiments_names:
    print("Loading results for experiment:", experiment_name)
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", experiment_results.latency_p50)
    print("Latency p99:", experiment_results.latency_p99)
    print("Token Usage:", experiment_results.total_tokens)
    print("Feedback Stats:", experiment_results.feedback_stats)
    print("*" * 50)

Loading results for experiment: Human-Eval-Code-5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-d4a27598
Latency p50: 0:00:14.059000
Latency p99: 0:01:33.983920
Token Usage: 30497
Feedback Stats: {'conciseness': {'n': 5, 'avg': 0.8, 'stdev': 0.39999999999999997, 'errors': 0, 'values': {}, 'type': 'primary'}, 'correctness': {'n': 5, 'avg': 0.2, 'stdev': 0.4, 'errors': 0, 'values': {}, 'type': 'primary'}}
**************************************************
Loading results for experiment: Human-Eval-Code-5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-696fcfd1
Latency p50: 0:00:28.352000
Latency p99: 0:00:37.063640
Token Usage: 14098
Feedback Stats: {'conciseness': {'n': 5, 'avg': 1.0, 'stdev': 0.0, 'errors': 0, 'values': {}, 'type': 'primary'}, 'correctness': {'n': 5, 'avg': 0.2, 'stdev': 0.4, 'errors': 0, 'values': {}, 'type': 'primary'}}
**************************************************
Loading results for experiment: Human-Eval-Code-5-aleatorios-mistralai/mistral-large-2-instruct-nvi

In [ ]:
experiment_results 

TracerSessionResult(id=UUID('88ae5449-8a38-4f2c-993a-109ed7310315'), start_time=datetime.datetime(2025, 10, 10, 12, 15, 21, 731723, tzinfo=datetime.timezone.utc), end_time=None, description=None, name='Human-Eval-Code-5-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-dfe04dc5', extra={'metadata': {'git': {'tags': None, 'dirty': True, 'branch': 'secundario', 'commit': 'b6fc7aa084265ccdf6ca0e7ce57ad7cba225e269', 'repo_name': 'AgenteCodificaoLangGraph', 'remote_url': 'https://github.com/Jeferson100/Code-Agent.git', 'author_name': 'JEFERSON DIONEI SEHNEM', 'commit_time': '1760031455', 'author_email': 'sehnemjeferson@gmail.com'}, 'revision_id': 'b6fc7aa-dirty', 'dataset_splits': ['base'], 'dataset_version': '2025-10-09T12:51:22.726379+00:00', 'num_repetitions': 1}}, tenant_id=UUID('d9ab5018-45a8-5efd-961d-320c87839c87'), reference_dataset_id=UUID('4dfd7fc2-d782-4c4d-ba17-87f5a251f9a7'), run_count=5, latency_p50=datetime.timedelta(seconds=93, microseconds=388000), latency_p99=datetime.tim

In [ ]:
import pandas as pd
data_frames = pd.DataFrame()
for experiment_name in experiments_names:
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    name_column = experiment_name.split("Human-Eval-Code-")[1]
    data = pd.DataFrame.from_dict(experiment_results.dict()).T
    try:
        data_frames[name_column] = data["correctness"]
    except Exception as e:
        print(e)
        pass
    

In [ ]:
data_frames

,5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-d4a27598,5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-696fcfd1,5-aleatorios-mistralai/mistral-large-2-instruct-nvidia-e1645f1c,5-aleatorios-openai/gpt-oss-20b-nvidia-8a4f22b3,5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-b3fcb57f,5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-7665a03d,5-aleatorios-mistralai/mistral-nemotron-nvidia-c56c3316,5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-bc289731,5-aleatorios-qwen/qwq-32b-nvidia-30b9c36d,5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-3814948e,...,5-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-c0f1f1dd,5-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1.5-nvidia-2ad25d4c,5-aleatorios-meta/llama-3.3-70b-instruct-nvidia-c40157c1,5-aleatorios-moonshotai/kimi-k2-instruct-nvidia-ba8a0136,5-aleatorios-openai/gpt-oss-120b-nvidia-ab182cb3,5-aleatorios-meta/llama-4-scout-17b-16e-instruct-nvidia-a8754be8,5-aleatorios-moonshotai/kimi-k2-instruct-0905-nvidia-475f09e9,5-aleatorios-microsoft/phi-4-mini-instruct-nvidia-95a90580,5-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-3967294a,5-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-dfe04dc5
id,693e3c6a-3b52-43fc-87c1-7d2f4f1eeecd,65eb49c9-dea7-47e4-81ff-4358de68e932,7ccac398-e870-4765-8847-902d60fc4d6b,d40eae6a-68f3-408e-873e-4ae2d5207da6,9461fb13-b4dc-49e5-b686-a93f6ead63ee,69dfe93c-b186-43da-bf53-dcfd28937aa9,f4b9bb32-0124-414c-ad9b-91889b11a0d2,73887851-8003-4eb5-b426-dc142aa4dd4f,dff60ec4-69fa-4310-8ecb-831aba7c271e,54e908ff-ca4b-49a2-bc61-4903d07278be,...,0091fad7-f9a1-40e7-ada0-68e86bf861e1,34729a40-ab1a-4b6f-bfc5-b3cde53e2952,2fb9e3c4-e5db-4784-8770-97f565a71450,85c23511-7aca-4ca4-a391-4392005576ea,b85d4408-426e-415e-a2f3-bcb16934b510,6f34b726-6650-4dff-ad0c-837730e7badd,35c28bf6-cc7c-4824-9ed5-6be3142911d1,e43f3ba7-67c9-4a4a-9566-a713413febd8,af710802-9a93-4c52-8b31-20f74e31ecee,88ae5449-8a38-4f2c-993a-109ed7310315
start_time,2025-10-13 13:30:38.526548+00:00,2025-10-13 13:27:37.093846+00:00,2025-10-13 13:26:56.491535+00:00,2025-10-13 13:24:26.958663+00:00,2025-10-13 13:04:57.972360+00:00,2025-10-13 12:12:48.396137+00:00,2025-10-13 12:11:58.632859+00:00,2025-10-10 19:14:49.408136+00:00,2025-10-10 19:14:04.820598+00:00,2025-10-10 19:11:53.234268+00:00,...,2025-10-10 13:06:16.473292+00:00,2025-10-10 13:04:21.693715+00:00,2025-10-10 13:01:33.679599+00:00,2025-10-10 12:58:26.332688+00:00,2025-10-10 12:54:04.241227+00:00,2025-10-10 12:51:19.010833+00:00,2025-10-10 12:49:38.326797+00:00,2025-10-10 12:47:49.039363+00:00,2025-10-10 12:27:08.721479+00:00,2025-10-10 12:15:21.731723+00:00
end_time,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
name,Human-Eval-Code-5-aleatorios-meta/llama-3.1-70...,Human-Eval-Code-5-aleatorios-deepseek-ai/deeps...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-openai/gpt-oss-20...,Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b...,Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b...,Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvid...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,...,Human-Eval-Code-5-aleatorios-nvidia/llama-3.1-...,Human-Eval-Code-5-aleatorios-nvidia/llama-3.3-...,Human-Eval-Code-5-aleatorios-meta/llama-3.3-70...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-openai/gpt-oss-12...,Human-Eval-Code-5-aleatorios-meta/llama-4-scou...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-microsoft/phi-4-m...,Human-Eval-Code-5-aleatorios-deepseek-ai/deeps...,Human-Eval-Code-5-aleatorios-qwen/qwen3-next-8...
extra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenant_id,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-3

### Acessando datasets do LangSmith

In [ ]:
# List all datasets
from langsmith import Client

client = Client()

datasets = client.list_datasets()
for dataset in datasets:
    print(dataset.id, dataset.name)

4dfd7fc2-d782-4c4d-ba17-87f5a251f9a7 Human-Eval-Code-5-aleatorios
b9dd1aaa-017b-4280-96ca-cd6f094773fa deep_research_supervisor_parallelism
5873a0fe-22fd-4a3d-8eb6-86330c26e52f deep_research_agent_termination
2bdcce58-182b-4a36-98a5-74720bc26a05 deep_research_scoping
27916b23-40e0-4874-8d5c-ddbebfcb8958 E-mail Triage Evaluation
b746e978-0744-44de-9f42-6cbd42bbd1e3 agents-from-scratch.test_response
623cff70-db0b-4993-bfe4-6ee88d1a46ac agents-from-scratch.test_tools
a2defcc0-e281-43c6-b351-c7930a9307ac Financial Advisory RAG Evaluation
77fa7a2a-4ece-4096-96fa-ed9d1d9cff81 Healthcare Agent Trajectory Evaluation
43be1111-6a51-4928-a33e-940a33a8b40a Reasoning and Bias
ba1609ff-70a6-4a64-b849-1892bbd24274 QA Example Dataset
aea62e89-af3d-4bb0-98e6-275fd643bfe5 Sample dataset
7f372826-0146-440b-b0e9-32a41cd01585 Human-Eval-Code-10-aleatorios
da848d07-6fa9-4ffe-a503-570e10840ac0 Insurance Claims


In [ ]:
# List one dataset by ID
dataset = client.read_dataset(dataset_id="0b278319-9299-4f5c-8ac3-d068c769469f")
print(dataset)


LangSmithNotFoundError: Resource not found for /datasets/0b278319-9299-4f5c-8ac3-d068c769469f. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/datasets/0b278319-9299-4f5c-8ac3-d068c769469f?limit=1', '{"detail":"Dataset not found"}')

In [ ]:
examples = client.list_examples(dataset_id="0b278319-9299-4f5c-8ac3-d068c769469f")
for example in examples:
    print(example.inputs, example.outputs)

{'aswer_code': 'from typing import List\n\n\ndef below_zero(operations: List[int]) -> bool:\n    """ You\'re given a list of deposit and withdrawal operations on a bank account that starts with\n    zero balance. Your task is to detect if at any point the balance of account fallls below zero, and\n    at that point function should return True. Otherwise it should return False.\n    >>> below_zero([1, 2, 3])\n    False\n    >>> below_zero([1, 2, -4, 5])\n    True\n    """\n'} {'response_code': "\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([]) == False\n    assert candidate([1, 2, -3, 1, 2, -3]) == False\n    assert candidate([1, 2, -4, 5, 6]) == True\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -4]) == False\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -5]) == True\n    assert candidate([1, -2, 2, -2, 5, -5, 4, -4]) == True\n"}
{'aswer_code': 'from typing import List\n\n\ndef separate_paren_groups(paren_string: str)

In [ ]:
experiment_results.dict()

{'id': UUID('e9d43d28-170e-4122-9546-956cf5dec521'),
 'start_time': datetime.datetime(2025, 10, 3, 15, 18, 51, 697051, tzinfo=datetime.timezone.utc),
 'end_time': None,
 'description': None,
 'name': 'test-human-eval-5-qwen/qwen3-next-80b-a3b-instruct-nvidia-dc157e0e',
 'extra': {'metadata': {'git': {'tags': None,
    'dirty': True,
    'branch': 'secundario',
    'commit': '47b18b6248c690a39e12ddf4182201433d49be46',
    'repo_name': 'AgenteCodificaoLangGraph',
    'remote_url': 'https://github.com/Jeferson100/Code-Agent.git',
    'author_name': 'JEFERSON DIONEI SEHNEM',
    'commit_time': '1759163044',
    'author_email': 'sehnemjeferson@gmail.com'},
   'revision_id': '47b18b6-dirty',
   'dataset_splits': ['base'],
   'dataset_version': '2025-10-02T13:48:42.613843+00:00',
   'num_repetitions': 1}},
 'tenant_id': UUID('d9ab5018-45a8-5efd-961d-320c87839c87'),
 'reference_dataset_id': UUID('7556c14d-cd6e-4aaf-9bec-979228fa1f71'),
 'run_count': 5,
 'latency_p50': datetime.timedelta(second